# Greeks Analysis and Hedging Strategies

This notebook explores option Greeks (sensitivities) and their applications in hedging.

## Topics Covered:
1. Greeks Calculation (Monte Carlo vs Black-Scholes)
2. Greeks Surfaces
3. Delta Hedging Strategy
4. Gamma Risk Analysis
5. Vega and Volatility Risk

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
import sys
sys.path.insert(0, '../src')

from mc_pricing import (
    EuropeanOption, BlackScholes,
    MonteCarloSimulator, Visualizer
)
from mc_pricing.greeks import FiniteDifferenceGreeks, PathwiseGreeks

%matplotlib inline
plt.style.use('seaborn-v0_8')

## 1. Calculate All Greeks

In [ ]:
# Define ATM call option
option = EuropeanOption(
    option_type='call',
    strike=100.0,
    maturity=1.0,
    spot=100.0,
    rate=0.05,
    volatility=0.2
)

# Black-Scholes Greeks
bs_greeks = BlackScholes.greeks(
    option.option_type,
    option.spot,
    option.strike,
    option.maturity,
    option.rate,
    option.volatility
)

print("Black-Scholes Greeks:")
print(f"  Delta: {bs_greeks['delta']:.4f}")
print(f"  Gamma: {bs_greeks['gamma']:.4f}")
print(f"  Vega:  {bs_greeks['vega']:.4f}")
print(f"  Theta: {bs_greeks['theta']:.4f}")
print(f"  Rho:   {bs_greeks['rho']:.4f}")

In [ ]:
# Monte Carlo Greeks (Finite Difference)
simulator = MonteCarloSimulator(n_simulations=100000, seed=42)
fd_greeks = FiniteDifferenceGreeks(simulator)

print("\nMonte Carlo Greeks (Finite Difference):")
mc_delta = fd_greeks.delta(option)
mc_gamma = fd_greeks.gamma(option)
mc_vega = fd_greeks.vega(option)
mc_theta = fd_greeks.theta(option)
mc_rho = fd_greeks.rho(option)

print(f"  Delta: {mc_delta:.4f} (error: {abs(mc_delta - bs_greeks['delta']):.4f})")
print(f"  Gamma: {mc_gamma:.4f} (error: {abs(mc_gamma - bs_greeks['gamma']):.4f})")
print(f"  Vega:  {mc_vega:.4f} (error: {abs(mc_vega - bs_greeks['vega']):.4f})")
print(f"  Theta: {mc_theta:.4f} (error: {abs(mc_theta - bs_greeks['theta']):.4f})")
print(f"  Rho:   {mc_rho:.4f} (error: {abs(mc_rho - bs_greeks['rho']):.4f})")

In [ ]:
# Pathwise Greeks (more efficient for Delta and Vega)
pw_greeks = PathwiseGreeks(n_simulations=100000, seed=42)

print("\nMonte Carlo Greeks (Pathwise):")
pw_delta = pw_greeks.delta(option)
pw_vega = pw_greeks.vega(option)

print(f"  Delta: {pw_delta:.4f} (error: {abs(pw_delta - bs_greeks['delta']):.4f})")
print(f"  Vega:  {pw_vega:.4f} (error: {abs(pw_vega - bs_greeks['vega']):.4f})")

## 2. Delta Surface

Visualize how Delta changes with spot price and time to maturity.

In [ ]:
# Create Delta surface
spots = np.linspace(80, 120, 20)
maturities = np.linspace(0.1, 2.0, 20)

delta_surface = np.zeros((len(maturities), len(spots)))

for i, T in enumerate(maturities):
    for j, S in enumerate(spots):
        greeks = BlackScholes.greeks('call', S, 100, T, 0.05, 0.2)
        delta_surface[i, j] = greeks['delta']

# Plot 3D surface
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

S, T = np.meshgrid(spots, maturities)
surf = ax.plot_surface(S, T, delta_surface, cmap='viridis', alpha=0.9)

ax.set_xlabel('Spot Price ($)', fontsize=11)
ax.set_ylabel('Time to Maturity (years)', fontsize=11)
ax.set_zlabel('Delta', fontsize=11)
ax.set_title('Call Option Delta Surface (K=100)', fontsize=14, fontweight='bold')

fig.colorbar(surf, shrink=0.5, aspect=5)
plt.tight_layout()
plt.show()

print("Key observations:")
print("- Delta approaches 1.0 for deep ITM options")
print("- Delta approaches 0.0 for deep OTM options")
print("- ATM delta ~0.5-0.6 (depends on time to maturity)")

## 3. Gamma Surface

Gamma measures the rate of change of Delta.

In [ ]:
# Create Gamma surface
gamma_surface = np.zeros((len(maturities), len(spots)))

for i, T in enumerate(maturities):
    for j, S in enumerate(spots):
        greeks = BlackScholes.greeks('call', S, 100, T, 0.05, 0.2)
        gamma_surface[i, j] = greeks['gamma']

# Plot 3D surface
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(S, T, gamma_surface, cmap='plasma', alpha=0.9)

ax.set_xlabel('Spot Price ($)', fontsize=11)
ax.set_ylabel('Time to Maturity (years)', fontsize=11)
ax.set_zlabel('Gamma', fontsize=11)
ax.set_title('Call Option Gamma Surface (K=100)', fontsize=14, fontweight='bold')

fig.colorbar(surf, shrink=0.5, aspect=5)
plt.tight_layout()
plt.show()

print("Key observations:")
print("- Gamma is highest for ATM options")
print("- Gamma increases as expiration approaches")
print("- Gamma is approximately zero for deep ITM/OTM")

## 4. Delta Hedging Strategy

Simulate a delta-hedged portfolio over time.

In [ ]:
# Simulation parameters
np.random.seed(42)
initial_spot = 100.0
strike = 100.0
maturity = 1.0
rate = 0.05
volatility = 0.2
n_steps = 252  # Daily rehedging
dt = maturity / n_steps

# Simulate price path
price_path = [initial_spot]
for _ in range(n_steps):
    dW = np.random.randn() * np.sqrt(dt)
    S_new = price_path[-1] * np.exp((rate - 0.5*volatility**2)*dt + volatility*dW)
    price_path.append(S_new)

price_path = np.array(price_path)
time_path = np.linspace(0, maturity, n_steps + 1)

# Calculate delta hedge at each step
option_values = []
deltas = []
hedge_errors = []

for i, (S, t) in enumerate(zip(price_path, time_path)):
    T_remaining = maturity - t
    
    if T_remaining > 0:
        # Option value
        V = BlackScholes.price('call', S, strike, T_remaining, rate, volatility)
        # Delta
        greeks = BlackScholes.greeks('call', S, strike, T_remaining, rate, volatility)
        delta = greeks['delta']
    else:
        # At expiration
        V = max(S - strike, 0)
        delta = 1.0 if S > strike else 0.0
    
    option_values.append(V)
    deltas.append(delta)

# Plot results
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Price path
axes[0, 0].plot(time_path, price_path, linewidth=2)
axes[0, 0].axhline(y=strike, color='r', linestyle='--', label='Strike')
axes[0, 0].set_xlabel('Time (years)')
axes[0, 0].set_ylabel('Spot Price ($)')
axes[0, 0].set_title('Underlying Asset Price Path')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Option value
axes[0, 1].plot(time_path, option_values, linewidth=2, color='green')
axes[0, 1].set_xlabel('Time (years)')
axes[0, 1].set_ylabel('Option Value ($)')
axes[0, 1].set_title('Call Option Value Over Time')
axes[0, 1].grid(True, alpha=0.3)

# Delta hedge ratio
axes[1, 0].plot(time_path, deltas, linewidth=2, color='orange')
axes[1, 0].set_xlabel('Time (years)')
axes[1, 0].set_ylabel('Delta')
axes[1, 0].set_title('Delta Hedge Ratio')
axes[1, 0].set_ylim(-0.1, 1.1)
axes[1, 0].grid(True, alpha=0.3)

# Hedged portfolio value
# Portfolio = Long option, Short delta shares
portfolio_value = np.array(option_values) - np.array(deltas) * price_path
axes[1, 1].plot(time_path, portfolio_value, linewidth=2, color='purple')
axes[1, 1].set_xlabel('Time (years)')
axes[1, 1].set_ylabel('Portfolio Value ($)')
axes[1, 1].set_title('Delta-Hedged Portfolio')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal spot price: ${price_path[-1]:.2f}")
print(f"Final option payoff: ${max(price_path[-1] - strike, 0):.2f}")
print(f"Portfolio volatility: ${np.std(portfolio_value):.4f}")

## 5. Vega Profile Across Strikes

Analyze volatility sensitivity for different moneyness levels.

In [ ]:
# Vega across strikes
spot = 100
strikes = np.linspace(70, 130, 50)
maturity = 1.0

vegas = []
for K in strikes:
    greeks = BlackScholes.greeks('call', spot, K, maturity, 0.05, 0.2)
    vegas.append(greeks['vega'])

plt.figure(figsize=(10, 6))
plt.plot(strikes, vegas, linewidth=2.5, color='teal')
plt.axvline(x=spot, color='r', linestyle='--', label='ATM', linewidth=2)
plt.xlabel('Strike Price ($)', fontsize=12)
plt.ylabel('Vega (per 1% vol change)', fontsize=12)
plt.title('Vega Profile Across Strikes (1Y Maturity)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Key observations:")
print("- Vega is highest for ATM options")
print("- OTM and ITM options have lower vega")
print("- Vega represents exposure to volatility risk")

## 6. Greeks Comparison Table

In [ ]:
# Create comprehensive Greeks table
moneyness_levels = [
    ('Deep ITM', 80),
    ('ITM', 90),
    ('ATM', 100),
    ('OTM', 110),
    ('Deep OTM', 120)
]

greeks_data = []

for name, strike in moneyness_levels:
    greeks = BlackScholes.greeks('call', 100, strike, 1.0, 0.05, 0.2)
    greeks_data.append({
        'Moneyness': name,
        'Strike': strike,
        'Delta': f"{greeks['delta']:.4f}",
        'Gamma': f"{greeks['gamma']:.4f}",
        'Vega': f"{greeks['vega']:.4f}",
        'Theta': f"{greeks['theta']:.4f}",
        'Rho': f"{greeks['rho']:.4f}"
    })

df = pd.DataFrame(greeks_data)
print("\nGreeks Across Moneyness Levels (Call Options):")
print(df.to_string(index=False))

# Interpretation guide
print("\n" + "="*70)
print("Greeks Interpretation Guide:")
print("="*70)
print("Delta:  Hedge ratio, ∂V/∂S (0 to 1 for calls)")
print("Gamma:  Curvature, ∂²V/∂S² (highest for ATM)")
print("Vega:   Vol sensitivity, ∂V/∂σ (highest for ATM)")
print("Theta:  Time decay, ∂V/∂t (negative for long options)")
print("Rho:    Rate sensitivity, ∂V/∂r")

## 7. Practical Hedging Strategy

Example: Portfolio with multiple positions.

In [ ]:
# Portfolio positions
portfolio = [
    {'name': 'Long Call 95', 'type': 'call', 'strike': 95, 'quantity': 10},
    {'name': 'Short Call 105', 'type': 'call', 'strike': 105, 'quantity': -10},
    {'name': 'Long Put 95', 'type': 'put', 'strike': 95, 'quantity': 5}
]

spot = 100
maturity = 0.5
rate = 0.05
volatility = 0.25

# Calculate portfolio Greeks
total_delta = 0
total_gamma = 0
total_vega = 0
total_theta = 0

print("Portfolio Greeks Analysis:")
print(f"{'Position':<20} {'Qty':<8} {'Delta':<10} {'Gamma':<10} {'Vega':<10} {'Theta':<10}")
print("-" * 75)

for pos in portfolio:
    greeks = BlackScholes.greeks(
        pos['type'], spot, pos['strike'], maturity, rate, volatility
    )
    
    delta = greeks['delta'] * pos['quantity']
    gamma = greeks['gamma'] * pos['quantity']
    vega = greeks['vega'] * pos['quantity']
    theta = greeks['theta'] * pos['quantity']
    
    print(f"{pos['name']:<20} {pos['quantity']:<8} {delta:<10.2f} {gamma:<10.4f} "
          f"{vega:<10.2f} {theta:<10.4f}")
    
    total_delta += delta
    total_gamma += gamma
    total_vega += vega
    total_theta += theta

print("-" * 75)
print(f"{'TOTAL':<20} {'':8} {total_delta:<10.2f} {total_gamma:<10.4f} "
      f"{total_vega:<10.2f} {total_theta:<10.4f}")

# Hedge recommendation
print("\nHedging Recommendations:")
print(f"- Need to {'sell' if total_delta > 0 else 'buy'} {abs(total_delta):.0f} shares to delta hedge")
print(f"- Portfolio has {'positive' if total_gamma > 0 else 'negative'} gamma: "
      f"{abs(total_gamma):.4f}")
print(f"- Vega exposure: {total_vega:.2f} (profit from vol increase)" if total_vega > 0 
      else f"- Vega exposure: {total_vega:.2f} (loss from vol increase)")
print(f"- Daily theta decay: ${total_theta:.2f}")

## Summary

### Key Takeaways:

1. **Delta**: Primary hedging Greek, rebalance frequently
2. **Gamma**: Measures delta instability, highest near ATM
3. **Vega**: Volatility exposure, important for vol trading
4. **Theta**: Time decay, always negative for long positions
5. **Monte Carlo vs Analytical**: MC within 1-2% of BS for most Greeks

### Best Practices:
- Delta hedge daily for large positions
- Monitor gamma risk near expiration
- Vega hedge if managing volatility risk
- Use pathwise method for faster Delta/Vega calculation

Next: Check out `04_convergence_study.ipynb` for Monte Carlo convergence analysis!